In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import scipy.stats as stats
from matplotlib.ticker import MaxNLocator, MultipleLocator
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. Simulation Settings
# ==========================================
num_simulations = 999 
c_resolution = 300
c_max = 40.0
# X-axis range 0 to 5
c_plot_xlim = (0, 5)

dt = 0.5
total_time = 12
time_steps = int(total_time / dt)

# ==========================================
# 2. Biological Parameters
# ==========================================
S0_total = 1e7
R0_total = 1e6 

d = 0.07
K_cap = 1e9

Vsmin = -0.1
Vrmin = -0.1

Vsmax_range = (0.98, 1.0)
Vrmax_range = (0.89, 0.90)

MICs_range = (2.0, 8.0)
Ks_range   = (4.0, 6.0)

MICr_range = (64.0, 256.0)
Kr_range   = (1.0, 2.0)

SC_CAP = 10.0 

# ==========================================
# 3. H_all: Overall Community Shannon Diversity
# ==========================================
def shannon_H_all(i_s, j_r, S_tot=S0_total, R_tot=R0_total):
    T = S_tot + R_tot
    pS = (S_tot / T) * (1.0 / i_s)
    pR = (R_tot / T) * (1.0 / j_r)
    return -(i_s * pS * np.log(pS) + j_r * pR * np.log(pR))

def H_theoretical_range_from_ij_box(i_min, i_max, j_min, j_max):
    H_vals = []
    for i in range(i_min, i_max + 1):
        for j in range(j_min, j_max + 1):
            H_vals.append(shannon_H_all(i, j))
    H_vals = np.array(H_vals, dtype=float)
    return float(np.min(H_vals)), float(np.max(H_vals))

def ci95_margin(std_dev, n):
    se = std_dev / np.sqrt(n)
    t_critical = stats.t.ppf(0.975, df=n - 1)
    return t_critical * se

# ==========================================
# 4. Explicit Extreme Value Effect
# ==========================================
I_GLOBAL_MIN, I_GLOBAL_MAX = 5, 50
EXTREME_ALPHA = 3.0 

def sample_MICs_extreme_high(i_s, rng, low=2.0, high=8.0, alpha=3.0):
    x = (i_s - I_GLOBAL_MIN) / (I_GLOBAL_MAX - I_GLOBAL_MIN)
    x = float(np.clip(x, 0.0, 1.0))
    pool_size = int(np.ceil(i_s * (1.0 + alpha * x)))
    pool = rng.uniform(low, high, pool_size)
    mic = np.sort(pool)[-i_s:] 
    return mic

# ==========================================
# 5. Single Simulation Function
# ==========================================
def simulate_sc_curve_and_msc(
    c_range,
    i_s, j_r,
    MICs, Ks, Vsmax,
    MICr, Kr, Vrmax,
    gamma_ia, gamma_jb, gamma_ja, gamma_ib
):
    S0 = S0_total / i_s
    R0 = R0_total / j_r

    SC_curve = []

    for c in c_range:
        S = np.full(i_s, S0, dtype=float)
        R = np.full(j_r, R0, dtype=float)

        term_s = (c / MICs) ** Ks
        Vs = Vsmax - ((Vsmax - Vsmin) * term_s) / (term_s - (Vsmin / Vsmax))

        term_r = (c / MICr) ** Kr
        Vr = Vrmax - ((Vrmax - Vrmin) * term_r) / (term_r - (Vrmin / Vrmax))

        for _ in range(time_steps):
            total_S = np.sum(S)
            total_R = np.sum(R)

            comp_s = 1 - (gamma_ia * total_S + gamma_ja * total_R) / K_cap
            comp_r = 1 - (gamma_ib * total_S + gamma_jb * total_R) / K_cap

            S += (Vs * S * comp_s - d * S) * dt
            R += (Vr * R * comp_r - d * R) * dt

            S[S < 0] = 0
            R[R < 0] = 0

        total_S_end = np.sum(S)
        total_R_end = np.sum(R)
        total_S_start = i_s * S0
        total_R_start = j_r * R0

        if total_R_end < 1.0:
            sc_val = SC_CAP
        else:
            sc_val = (total_S_end / total_S_start) / (total_R_end / total_R_start)

        if np.isinf(sc_val) or np.isnan(sc_val) or sc_val > SC_CAP:
            sc_val = SC_CAP

        SC_curve.append(sc_val)

    SC_curve = np.array(SC_curve, dtype=float)

    try:
        if np.min(SC_curve) < 1.0 and np.max(SC_curve) > 1.0:
            msc_val = np.interp(1.0, SC_curve[::-1], c_range[::-1])
        else:
            msc_val = np.nan
    except:
        msc_val = np.nan

    return SC_curve, msc_val

# ==========================================
# 6. Study A: H_all Grouping Analysis
# ==========================================
DELTA_MIN, DELTA_MAX = 0.02, 0.12
GAMMA_IB_CAP = 1.40
EPS = 1e-6

def run_study_A_H_groups(ij_groups, seed_base=10101):
    c_range = np.linspace(0, c_max, c_resolution)
    results = {}

    print(f"\n--- Study A: H_all groups (linked i-j; extreme MICs) (N={num_simulations}) ---")

    for gid, g in enumerate(ij_groups):
        name = g["name"]
        i_min, i_max = g["i_range"]
        j_min, j_max = g["j_range"]

        H_min_theory, H_max_theory = H_theoretical_range_from_ij_box(i_min, i_max, j_min, j_max)

        msc_list, H_list, i_list, j_list = [], [], [], []
        curves = []

        print(f"   Processing {name} (i:{i_min}-{i_max}, j:{j_min}-{j_max}) ...")
        for sim in range(num_simulations):
            if sim % 100 == 0:
                print(f"     -> Simulating {sim}/{num_simulations}...", end="\r")

            rng = np.random.default_rng(seed_base + gid * 100000 + sim)

            i_s = int(rng.integers(i_min, i_max + 1))
            j_r = int(rng.integers(j_min, j_max + 1))

            gamma_intra = float(rng.uniform(0.95, 1.05)) 
            gamma_ja = float(rng.uniform(0.80, 1.20))
            delta = float(rng.uniform(DELTA_MIN, DELTA_MAX))
            gamma_ib = min(gamma_ja + delta, GAMMA_IB_CAP)
            if gamma_ib <= gamma_ja:
                gamma_ib = gamma_ja + EPS

            gamma_ia = gamma_intra
            gamma_jb = gamma_intra

            i_list.append(i_s)
            j_list.append(j_r)
            H_list.append(shannon_H_all(i_s, j_r))

            MICs  = sample_MICs_extreme_high(i_s, rng, low=MICs_range[0], high=MICs_range[1], alpha=EXTREME_ALPHA)
            Ks    = rng.uniform(Ks_range[0],    Ks_range[1],    i_s)
            Vsmax = rng.uniform(Vsmax_range[0], Vsmax_range[1], i_s)

            MICr  = rng.uniform(MICr_range[0], MICr_range[1], j_r)
            Kr    = rng.uniform(Kr_range[0],    Kr_range[1],    j_r)
            Vrmax = rng.uniform(Vrmax_range[0], Vrmax_range[1], j_r)

            SC_curve, msc_val = simulate_sc_curve_and_msc(
                c_range, i_s, j_r,
                MICs, Ks, Vsmax,
                MICr, Kr, Vrmax,
                gamma_ia, gamma_jb, gamma_ja, gamma_ib
            )

            curves.append(SC_curve)
            msc_list.append(msc_val)

        print("     -> Done.                                        ")

        curves = np.array(curves)
        results[name] = {
            "msc": np.array(msc_list, dtype=float),
            "H_all": np.array(H_list, dtype=float),
            "i_s": np.array(i_list, dtype=int),
            "j_r": np.array(j_list, dtype=int),
            "sc_mean": np.nanmean(curves, axis=0),
            "sc_std": np.nanstd(curves, axis=0),
            "c_axis": c_range,
            "H_min_theory": H_min_theory,
            "H_max_theory": H_max_theory,
        }

    return results

# ==========================================
# 7. Study B: Competition Intensity (Stratified Sampling)
# ==========================================
def run_study_B_competition_using_A_ijlink_stratified(
    gamma_ja_groups, ij_groups_like_A, seed_base=20202
):
    c_range = np.linspace(0, c_max, c_resolution)
    results = {}

    print(f"\n--- Study B: Competition (3 groups; Stratified 333x3; extreme MICs) (N={num_simulations}) ---")

    for gid, (gmin, gmax) in enumerate(gamma_ja_groups):
        label = f"g:{gmin:.2f}-{gmax:.2f}"

        msc_list = []
        curves = []

        print(f"   Processing gamma_ja in [{gmin}, {gmax}] ...")
        for sim in range(num_simulations):
            if sim % 100 == 0:
                print(f"     -> Simulating {sim}/{num_simulations}...", end="\r")

            rng = np.random.default_rng(seed_base + gid * 100000 + sim)

            g_idx = sim % len(ij_groups_like_A)
            gpick = ij_groups_like_A[g_idx]
            
            i_min, i_max = gpick["i_range"]
            j_min, j_max = gpick["j_range"]
            
            i_s = int(rng.integers(i_min, i_max + 1))
            j_r = int(rng.integers(j_min, j_max + 1))

            gamma_intra = float(rng.uniform(0.95, 1.05))

            gamma_ja = float(rng.uniform(gmin, gmax))
            delta = float(rng.uniform(DELTA_MIN, DELTA_MAX))
            gamma_ib = min(gamma_ja + delta, GAMMA_IB_CAP)
            if gamma_ib <= gamma_ja:
                gamma_ib = gamma_ja + EPS

            gamma_ia = gamma_intra
            gamma_jb = gamma_intra

            MICs  = sample_MICs_extreme_high(i_s, rng, low=MICs_range[0], high=MICs_range[1], alpha=EXTREME_ALPHA)
            Ks    = rng.uniform(Ks_range[0],    Ks_range[1],    i_s)
            Vsmax = rng.uniform(Vsmax_range[0], Vsmax_range[1], i_s)

            MICr  = rng.uniform(MICr_range[0], MICr_range[1], j_r)
            Kr    = rng.uniform(Kr_range[0],    Kr_range[1],    j_r)
            Vrmax = rng.uniform(Vrmax_range[0], Vrmax_range[1], j_r)

            SC_curve, msc_val = simulate_sc_curve_and_msc(
                c_range, i_s, j_r,
                MICs, Ks, Vsmax,
                MICr, Kr, Vrmax,
                gamma_ia, gamma_jb, gamma_ja, gamma_ib
            )

            curves.append(SC_curve)
            msc_list.append(msc_val)

        print("     -> Done.                                        ")

        curves = np.array(curves)
        results[label] = {
            "msc": np.array(msc_list, dtype=float),
            "sc_mean": np.nanmean(curves, axis=0),
            "sc_std": np.nanstd(curves, axis=0),
            "c_axis": c_range,
            "gmin": gmin,
            "gmax": gmax,
        }

    return results

# ==========================================
# 8. Group Settings
# ==========================================
ij_groups_A = [
    {"name": "Low-H",  "i_range": (5, 12),  "j_range": (1, 2)},
    {"name": "Mid-H",  "i_range": (13, 28), "j_range": (2, 3)},
    {"name": "High-H", "i_range": (29, 50), "j_range": (4, 5)},
]
gamma_ja_groups_B = [(0.80, 0.90), (0.95, 1.05), (1.10, 1.20)]

H_INTERVAL_LABELS = {
    "Low-H":  "1.77-2.63",
    "Mid-H":  "2.70-3.43",
    "High-H": "3.49-4.01",
}

# ==========================================
# 9. Execute Both Studies
# ==========================================
results_A = run_study_A_H_groups(ij_groups_A)
results_B = run_study_B_competition_using_A_ijlink_stratified(gamma_ja_groups_B, ij_groups_A)

# ==========================================
# 10. Export Simplified Excel
# ==========================================
save_dir = "./results_ecological"
os.makedirs(save_dir, exist_ok=True)

def export_excel_simple_fixed_labels(results_A, results_B, out_path):
    data = {}
    for name, res in results_A.items():
        hlabel = H_INTERVAL_LABELS.get(name, f"{res['H_min_theory']:.2f}-{res['H_max_theory']:.2f}")
        data[f"A_H_{hlabel}_MSC"] = res["msc"]

    for label, res in results_B.items():
        data[f"B_gammaja_{res['gmin']:.2f}-{res['gmax']:.2f}_MSC"] = res["msc"]

    df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
    df.to_excel(out_path, index_label="Sim_ID")
    print(f"\nSimplified MSC raw Excel saved to: {out_path}")

excel_path = os.path.join(save_dir, "MSC_raw_simple_stratified_999.xlsx")
export_excel_simple_fixed_labels(results_A, results_B, excel_path)

# ==========================================
# 11. Plotting (Oversized Fonts and Independent System)
# ==========================================
plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams['mathtext.fontset'] = 'stix'

# --- Style Configuration (Oversized System) ---
CONFIG = {
    'label_size': 55,        # Axis title font size
    'tick_size': 54,         # Axis tick number size
    'legend_size': 36,       # Legend font size
    'annotation_size': 45,   # "SC = 1" annotation size
    
    'border_width': 5,       # Axis border and tick mark linewidth
    'plot_linewidth': 5,     # Curve linewidth
    'ref_line_width': 6,     # SC=1 horizontal reference linewidth
    'ref_line_color': 'black', # Reference line color
    
    'bins': 200, 
    'major_tick_length': 16, # Major tick length
    
    'single_fig_size': (18, 24), # Width and height of a single split figure
    'y_label_x_pos': -0.14   # Y-axis title offset
}

# Maintain green/purple color palette
colors_A = ["#A1D99B", "#41AB5D", "#005A32"]
colors_B = ["#BCBDDC", "#9E9AC8", "#6A51A3"]

HEADROOM = 0.08 # Maintain 8% headroom
SC_LABEL_OFFSET_RATIO = 0.03

def apply_global_style(axes_list):
    """Apply global oversized ticks and border styles"""
    for ax in axes_list:
        ax.tick_params(axis='both', which='major',
                       labelsize=CONFIG['tick_size'],
                       width=CONFIG['border_width'],
                       length=CONFIG['major_tick_length'])
        for spine in ax.spines.values():
            spine.set_linewidth(CONFIG['border_width'])

def add_sc1_label(ax, x_text=4.9):
    ymin, ymax = ax.get_ylim()
    yr = ymax - ymin if ymax > ymin else 1.0
    y_text = 1.0 + SC_LABEL_OFFSET_RATIO * yr
    if y_text >= ymax - 0.01 * yr:
        ax.set_ylim(ymin, y_text + 0.08 * yr)
    ax.text(
        x_text, y_text, "SC = 1",
        fontsize=CONFIG["annotation_size"],
        color=CONFIG["ref_line_color"],
        ha="right", weight="bold", va="bottom"
    )

# ----------------------------------------------------
# Plot Figure 1: Study A (H) independent figure
# ----------------------------------------------------
fig_A, (axA_top, axA_bot) = plt.subplots(
    2, 1, figsize=CONFIG['single_fig_size'], dpi=100, 
    gridspec_kw={'height_ratios': [1, 1], 'hspace': 0}
)

global_max_y_A = 0

for idx, (name, res) in enumerate(results_A.items()):
    c = res["c_axis"]
    mean = res["sc_mean"]
    std = res["sc_std"]
    color = colors_A[idx]

    # Keep numbers outside the label to prevent math mode interference
    legend_label = rf"$H$: {res['H_min_theory']:.2f}-{res['H_max_theory']:.2f}"
    axA_top.plot(c, mean, color=color, linewidth=CONFIG["plot_linewidth"], label=legend_label)

    margin = ci95_margin(std, num_simulations)
    axA_top.fill_between(c, np.maximum(mean - margin, 0), mean + margin, color=color, alpha=0.4, edgecolor=None)

    valid_mask = np.isfinite(res["msc"]) & (res["msc"] > 0)
    clean = res["msc"][valid_mask]
    if len(clean) > 0:
        n_hist, _, _ = axA_bot.hist(
            clean, bins=CONFIG["bins"], range=c_plot_xlim,
            color=color, alpha=0.5, edgecolor=color, linewidth=0.8
        )
        global_max_y_A = max(global_max_y_A, np.max(n_hist))

# Formatting A
axA_top.axhline(1.0, color=CONFIG["ref_line_color"], linestyle="--", linewidth=CONFIG["ref_line_width"], alpha=0.8)
axA_top.set_ylabel("Selection Coefficient (SC)", fontsize=CONFIG["label_size"])

# [Core modification] Force legend to use Arial font
axA_top.legend(loc="upper right", frameon=False, prop={'family': 'Arial', 'size': CONFIG['legend_size']})

axA_top.margins(y=0.06)
add_sc1_label(axA_top, x_text=4.9)

for ax in [axA_top, axA_bot]:
    ax.set_xlim(*c_plot_xlim)
    ax.xaxis.set_major_locator(MultipleLocator(1))

apply_global_style([axA_top, axA_bot])

axA_top.set_xticklabels([])
axA_bot.set_xlabel("Antibiotic concentration (c)", fontsize=CONFIG["label_size"], labelpad=15)
axA_bot.set_ylabel("Frequency", fontsize=CONFIG["label_size"])

if global_max_y_A > 0:
    limit_A = global_max_y_A * (1.0 + HEADROOM)
    axA_bot.set_ylim(0, limit_A)
    axA_bot.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))

fig_A.align_ylabels([axA_top, axA_bot])
axA_top.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)
axA_bot.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)

img_path_A = os.path.join(save_dir, "StudyA_H_stratified_curves.png")
try:
    fig_A.savefig(img_path_A, dpi=600, bbox_inches="tight")
    print(f"Independent Study A figure saved to: {img_path_A}")
except Exception as e:
    print(f"Failed to save Study A figure: {e}")


# ----------------------------------------------------
# Plot Figure 2: Study B (Gamma) independent figure
# ----------------------------------------------------
fig_B, (axB_top, axB_bot) = plt.subplots(
    2, 1, figsize=CONFIG['single_fig_size'], dpi=100, 
    gridspec_kw={'height_ratios': [1, 1], 'hspace': 0}
)

global_max_y_B = 0

for idx, (label, res) in enumerate(results_B.items()):
    c = res["c_axis"]
    mean = res["sc_mean"]
    std = res["sc_std"]
    color = colors_B[idx]

    # Keep numbers outside the label to prevent math mode interference
    legend_label = rf"$\gamma_{{ja}}$: {res['gmin']:.2f}-{res['gmax']:.2f}"
    axB_top.plot(c, mean, color=color, linewidth=CONFIG["plot_linewidth"], label=legend_label)

    margin = ci95_margin(std, num_simulations)
    axB_top.fill_between(c, np.maximum(mean - margin, 0), mean + margin, color=color, alpha=0.4, edgecolor=None)

    valid_mask = np.isfinite(res["msc"]) & (res["msc"] > 0)
    clean = res["msc"][valid_mask]
    if len(clean) > 0:
        n_hist, _, _ = axB_bot.hist(
            clean, bins=CONFIG["bins"], range=c_plot_xlim,
            color=color, alpha=0.5, edgecolor=color, linewidth=0.8
        )
        global_max_y_B = max(global_max_y_B, np.max(n_hist))

# Formatting B
axB_top.axhline(1.0, color=CONFIG["ref_line_color"], linestyle="--", linewidth=CONFIG["ref_line_width"], alpha=0.8)
axB_top.set_ylabel("Selection Coefficient (SC)", fontsize=CONFIG["label_size"])

# [Core modification] Force legend to use Arial font
axB_top.legend(loc="upper right", frameon=False, prop={'family': 'Arial', 'size': CONFIG['legend_size']})

axB_top.margins(y=0.06)
add_sc1_label(axB_top, x_text=4.9)

for ax in [axB_top, axB_bot]:
    ax.set_xlim(*c_plot_xlim)
    ax.xaxis.set_major_locator(MultipleLocator(1))

apply_global_style([axB_top, axB_bot])

axB_top.set_xticklabels([])
axB_bot.set_xlabel("Antibiotic concentration (c)", fontsize=CONFIG["label_size"], labelpad=15)
axB_bot.set_ylabel("Frequency", fontsize=CONFIG["label_size"])

if global_max_y_B > 0:
    limit_B = global_max_y_B * (1.0 + HEADROOM)
    axB_bot.set_ylim(0, limit_B)
    axB_bot.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))

fig_B.align_ylabels([axB_top, axB_bot])
axB_top.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)
axB_bot.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)

img_path_B = os.path.join(save_dir, "StudyB_gammaja_stratified_curves.png")
try:
    fig_B.savefig(img_path_B, dpi=600, bbox_inches="tight")
    print(f"Independent Study B figure saved to: {img_path_B}")
except Exception as e:
    print(f"Failed to save Study B figure: {e}")

# ==========================================
# 12. Display Plots
# ==========================================
plt.show()